# Similaridade entre documentos e versões

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/04_similaridade_documentos_e_versoes.ipynb)

O Notebook 03 produziu conjuntos de tokens, vetores de contagem e pesos TF-IDF.
Agora veremos que “semelhante” não é propriedade única do texto: cada medida
responde a uma relação diferente.

## 1. Escolher a medida pela pergunta

![Jaccard compara conjuntos, cosseno compara vetores, distância de edição compara sequências e a leitura próxima examina resultados divergentes.](imagens/04_escolha_metrica.svg)

| Pergunta | Representação | Medida inicial |
|---|---|---|
| Compartilham vocabulário? | conjunto de termos | Jaccard |
| Têm perfis de frequência parecidos? | vetor de contagens | cosseno |
| Compartilham termos distintivos? | vetor TF-IDF | cosseno |
| Quanto uma versão precisa mudar? | sequência de caracteres ou tokens | edição |

Autores, períodos e coleções podem ser comparados agregando documentos, desde
que autoria e pertencimento estejam documentados. Nesta base não há campo de
autoria; portanto, não inventaremos autores para cumprir um exemplo.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import math
import re
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display

dados = pd.read_csv("dados/documentos_comparacao.csv")
dados["texto"] = dados["texto_comparacao"]
stopwords = {"a", "e", "o", "de", "do", "da", "em", "nas", "também"}

def tokenizar(texto):
    base = unicodedata.normalize("NFKD", texto.lower())
    base = "".join(c for c in base if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z]+", base) if t not in stopwords]

dados["tokens"] = dados["texto"].map(tokenizar)
matriz_contagens = pd.DataFrame(
    [Counter(t) for t in dados["tokens"]], index=dados["id_documento"]
).fillna(0).astype(int)
N = len(matriz_contagens)
df = matriz_contagens.gt(0).sum()
idf = ((N + 1) / (df + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)

Com as representações reconstruídas, começamos por ignorar frequências e
comparar somente presença e ausência de termos.

## 2. Similaridade de Jaccard

Para conjuntos de termos $A$ e $B$:

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|}.
$$

O resultado varia de 0 a 1. Repetir um termo não altera Jaccard nesta definição;
dois textos de tamanhos muito diferentes podem ser penalizados pela união ampla.

In [ ]:
def jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    return len(a & b) / len(a | b) if a | b else 1.0

consulta = "D001"
tokens_consulta = dados.set_index("id_documento").loc[consulta, "tokens"]
similaridades_jaccard = dados.set_index("id_documento")["tokens"].map(
    lambda tokens: jaccard(tokens_consulta, tokens)
).drop(consulta).sort_values(ascending=False)
similaridades_jaccard.head(6).rename("Jaccard")

Jaccard trata todos os termos presentes igualmente. Para preservar frequências
ou pesos, comparamos o ângulo entre vetores.

## 3. Similaridade de cosseno

$$
\cos(\mathbf{x},\mathbf{y})=
\frac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Em vetores não negativos, 1 indica mesma direção e 0 ausência de componentes
compartilhados. Cosseno reduz o efeito da magnitude total, mas continua
dependente do vocabulário, da ponderação e do pré-processamento.

In [ ]:
def cosseno(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    denominador = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denominador) if denominador else 0.0

vetor_contagem = matriz_contagens.loc[consulta]
vetor_tfidf = matriz_tfidf.loc[consulta]
similaridades_cosseno_contagem = matriz_contagens.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_contagem, linha), axis=1
).sort_values(ascending=False)
similaridades_cosseno_tfidf = matriz_tfidf.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_tfidf, linha), axis=1
).sort_values(ascending=False)
ranking = pd.concat([
    similaridades_jaccard.rename("Jaccard"),
    similaridades_cosseno_contagem.rename("cosseno contagens"),
    similaridades_cosseno_tfidf.rename("cosseno TF-IDF"),
], axis=1)
ranking.sort_values("cosseno TF-IDF", ascending=False).head(8).round(3)

Os três rankings podem divergir porque presença, repetição e raridade recebem
pesos diferentes. Essa divergência é material analítico, não um obstáculo a ser
escondido.

## 4. Identificar documentos semelhantes e inspecionar divergências

![Um documento consultado liga-se a vizinhos encontrados por Jaccard, cosseno de contagens e cosseno TF-IDF, incluindo um caso divergente.](imagens/04_vizinhos_documentais.svg)

Selecione pares que aparecem no topo de várias métricas e pares cuja posição
muda. Leia texto, metadados e extensão antes de chamá-los semelhantes.

In [ ]:
ids_inspecao = list(dict.fromkeys(
    [consulta]
    + similaridades_jaccard.head(2).index.tolist()
    + similaridades_cosseno_tfidf.head(2).index.tolist()
))
pares_inspecao = dados.set_index("id_documento").loc[
    ids_inspecao, ["ano", "genero", "local", "tema", "texto"]
]
display(ranking.loc[[i for i in ids_inspecao if i != consulta]].round(3))
pares_inspecao

Vetores são adequados para perfis lexicais. Para comparar versões quase iguais,
a ordem das unidades é crucial; passamos então à distância de edição.

## 5. Distância de edição e comparação de versões

A distância de Levenshtein é o menor número de inserções, exclusões e
substituições necessário para transformar uma sequência em outra:

$$
D_{i,j}=\min\begin{cases}
D_{i-1,j}+1 & \text{exclusão}\\
D_{i,j-1}+1 & \text{inserção}\\
D_{i-1,j-1}+\mathbf{1}(a_i\neq b_j) & \text{substituição}.
\end{cases}
$$

![Duas sequências curtas são alinhadas por operações de manter, substituir, inserir e excluir; a figura alerta que distância não interpreta o sentido.](imagens/04_distancia_edicao.svg)

A unidade pode ser caractere ou token. A distância bruta cresce com o tamanho;
uma versão normalizada divide pelo maior comprimento, com convenção declarada.

In [ ]:
def levenshtein(seq_a, seq_b):
    anterior = list(range(len(seq_b) + 1))
    for i, a in enumerate(seq_a, 1):
        atual = [i]
        for j, b in enumerate(seq_b, 1):
            atual.append(min(
                anterior[j] + 1,
                atual[j-1] + 1,
                anterior[j-1] + (a != b),
            ))
        anterior = atual
    return anterior[-1]

versoes = pd.read_csv("dados/versoes_textuais.csv")
resultados_edicao = []
for id_documento, grupo in versoes.groupby("id_documento"):
    x, y = grupo.iloc[0], grupo.iloc[1]
    tokens_x, tokens_y = tokenizar(x["texto"]), tokenizar(y["texto"])
    distancia = levenshtein(tokens_x, tokens_y)
    resultados_edicao.append({
        "id_documento": id_documento,
        "versão A": x["tipo_versao"], "versão B": y["tipo_versao"],
        "distância em tokens": distancia,
        "distância normalizada": distancia / max(len(tokens_x), len(tokens_y)),
    })
tabela_edicao = pd.DataFrame(resultados_edicao)
tabela_edicao.round(3)

A distância informa esforço mínimo de transformação, não gravidade editorial ou
mudança de sentido. Corrigir um nome próprio pode custar uma substituição e ser
substantivamente decisivo.

## 6. Comparar autores, períodos e coleções

Para grupos textuais, agregue somente após decidir se a unidade é documento,
autor ou coleção. Médias de vetores por período respondem a perfis médios; unir
todos os textos responde a um megadocumento e dá mais peso a grupos maiores.
Autoria não deve ser tratada como essência estilística: gênero, período,
transcrição e composição do corpus podem explicar parte da proximidade.

## Atividade integrada — relatório de pares

**Modalidade:** trios. **Tempo:** 35 minutos.

1. escolha um documento-consulta e três candidatos;
2. calcule Jaccard e cosseno com duas ponderações;
3. identifique uma concordância e uma divergência entre rankings;
4. leia os documentos e metadados;
5. para versões, calcule distância de edição e descreva as mudanças;
6. justifique qual medida responde melhor à pergunta.

**Pares, métricas e resultados:** Escreva aqui.

**Leitura próxima e escolha da medida:** Escreva aqui.

**O que a similaridade não demonstra:** Escreva aqui.

Leve o relatório à oficina. A entrega final exigirá uma medida principal, uma
alternativa de sensibilidade e casos que alterem ou qualifiquem a conclusão.